In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "How do POTS patients in the Long COVID community compare to the broader population?"

# POTS in Long COVID: A Subgroup Comparison

## Abstract

POTS (postural orthostatic tachycardia syndrome) is one of the most frequently identified conditions in the r/covidlonghaulers community, affecting 88 users in our one-month dataset. This analysis compares 54 POTS patients who filed treatment reports against 1,067 non-POTS reporters to determine whether POTS patients try more treatments, report worse outcomes, and respond differently to common therapies. POTS patients try nearly twice as many treatments (8.6 vs 4.4 mean drugs per user) yet report substantially lower positive rates (48% vs 62%, p<0.01). Treatment-level analysis reveals that several community favorites -- famotidine, nattokinase, cetirizine -- significantly underperform for POTS patients. Beta blockers, the clinical first-line treatment for POTS, show mixed community sentiment despite strong guideline support -- a paradox worth investigating. Conversely, magnesium (100% positive, n=7), electrolytes (88% positive, n=6), and N-acetylcysteine (86% positive, n=5) outperform their community-wide averages in POTS patients. These findings suggest that POTS patients face a harder treatment journey and that community-popular treatments do not uniformly transfer to this subgroup.


## 1. Data Exploration

Data covers: **2026-03-11 to 2026-04-10 (1 month)** from r/covidlonghaulers.

We define the POTS cohort as users with an extracted condition mention of "pots" or "dysautonomia" (a broader autonomic dysfunction category that subsumes POTS). The comparison group is all remaining users -- those with no POTS or dysautonomia label. This avoids the circular comparison pitfall of comparing POTS to the overall population (which contains POTS).

**Filtering:** Generic terms ("supplements," "medication," etc.) are excluded as non-actionable categories. Causal-context vaccines (covid vaccine, pfizer, moderna, booster, etc.) are excluded because their negative sentiment reflects perceived illness causation, not treatment failure. "Long covid" is excluded from condition co-occurrence charts since it defines the community itself.

In [ ]:

# ── Define POTS cohort ──
pots_users = pd.read_sql('''
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) IN ('pots', 'dysautonomia')
''', conn)
pots_ids = set(pots_users['user_id'])

# ── Exclusion sets ──
CAUSAL_NAMES = {
    'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine', 'vaccine injection',
    'pfizer', 'booster', 'moderna'
}

MERGE_MAP = {
    'pepcid': 'famotidine',
    'magnesium glycinate': 'magnesium',
    'zepbound': 'tirzepatide',
}

# ── All treatment reports with exclusions ──
all_tx = pd.read_sql('''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
''', conn)
all_tx['drug'] = all_tx['drug'].str.lower().replace(MERGE_MAP)
all_tx = all_tx[~all_tx['drug'].isin(GENERIC_TERMS)]
all_tx = all_tx[~all_tx['drug'].isin(CAUSAL_NAMES)]
all_tx['score'] = all_tx['sentiment'].map(SENTIMENT_SCORE)

# Split into POTS vs non-POTS
pots_tx = all_tx[all_tx['user_id'].isin(pots_ids)].copy()
non_pots_tx = all_tx[~all_tx['user_id'].isin(pots_ids)].copy()

# ── User-level summaries ──
def user_summary(df):
    return df.groupby('user_id').agg(
        n_drugs=('drug', 'nunique'),
        n_reports=('score', 'count'),
        avg_score=('score', 'mean'),
        pos_rate=('sentiment', lambda x: (x == 'positive').sum() / len(x))
    ).reset_index()

pots_summary = user_summary(pots_tx)
non_pots_summary = user_summary(non_pots_tx)

display(HTML(f'''
<div style="background:#f0f7ff; padding:15px; border-radius:8px; margin:10px 0;">
<h3 style="margin-top:0;">Dataset Overview</h3>
<table style="font-size:14px;">
<tr><td><b>Data covers:</b></td><td>2026-03-11 to 2026-04-10 (~1 month)</td></tr>
<tr><td><b>Total community users:</b></td><td>2,827</td></tr>
<tr><td><b>POTS/dysautonomia users:</b></td><td>{len(pots_ids)} ({len(pots_ids)/2827*100:.1f}% of community)</td></tr>
<tr><td><b>POTS users with treatment reports:</b></td><td>{len(pots_summary)}</td></tr>
<tr><td><b>Non-POTS users with treatment reports:</b></td><td>{len(non_pots_summary):,}</td></tr>
<tr><td><b>POTS mean treatments per user:</b></td><td>{pots_summary["n_drugs"].mean():.1f}</td></tr>
<tr><td><b>Non-POTS mean treatments per user:</b></td><td>{non_pots_summary["n_drugs"].mean():.1f}</td></tr>
</table>
</div>
'''))


The POTS cohort represents roughly 3% of the community but tries nearly double the number of treatments. This alone signals a more difficult treatment journey -- these patients are searching harder for relief.

## 2. Baseline Comparison: POTS vs Non-POTS Outcomes

Before examining individual treatments, we need to establish whether POTS patients report worse outcomes overall. This comparison is at the user level -- each person contributes one data point (their average sentiment across all treatments) to avoid users with many reports dominating the analysis.

In [ ]:

from scipy.stats import mannwhitneyu, fisher_exact
import math

# ── Mann-Whitney U on user-level average scores ──
u_stat, p_mw = mannwhitneyu(pots_summary['avg_score'], non_pots_summary['avg_score'], alternative='two-sided')
n1, n2 = len(pots_summary), len(non_pots_summary)
r_rb = 1 - (2 * u_stat) / (n1 * n2)  # rank-biserial correlation

# ── Fisher's exact on user-level positive outcome (>50% positive reports) ──
pots_pos_users = (pots_summary['pos_rate'] > 0.5).sum()
pots_neg_users = len(pots_summary) - pots_pos_users
np_pos_users = (non_pots_summary['pos_rate'] > 0.5).sum()
np_neg_users = len(non_pots_summary) - np_pos_users
table_2x2 = [[pots_pos_users, pots_neg_users], [np_pos_users, np_neg_users]]
or_val, p_fisher = fisher_exact(table_2x2)

# ── Cohen's h ──
p1 = pots_pos_users / n1
p2 = np_pos_users / n2
cohens_h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

# ── Grouped bar chart: outcome distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 1.5]})

# Panel A: Positive rate comparison with CIs
pots_k = pots_pos_users
np_k = np_pos_users
pots_ci = wilson_ci(pots_k, n1)
np_ci = wilson_ci(np_k, n2)

bars = axes[0].barh(['Non-POTS\n(n={:,})'.format(n2), 'POTS\n(n={})'.format(n1)],
                     [p2 * 100, p1 * 100],
                     color=[COLORS['positive'], '#e67e22'],
                     height=0.5, edgecolor='white')
# Error bars
axes[0].errorbar([p2 * 100, p1 * 100],
                 [0, 1],
                 xerr=[[p2*100 - np_ci[0]*100, p1*100 - pots_ci[0]*100],
                       [np_ci[1]*100 - p2*100, pots_ci[1]*100 - p1*100]],
                 fmt='none', color='black', capsize=5)
axes[0].set_xlabel('% Users with Majority-Positive Outcomes')
axes[0].set_title('Overall Outcome Rate')
axes[0].set_xlim(0, 100)
for bar, pct in zip(bars, [p2 * 100, p1 * 100]):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 f'{pct:.0f}%', va='center', fontsize=11, fontweight='bold')

# Panel B: Distribution of user-level avg scores
bins = np.linspace(-1, 1, 15)
axes[1].hist(non_pots_summary['avg_score'], bins=bins, alpha=0.6, label=f'Non-POTS (n={n2:,})',
             color=COLORS['positive'], density=True, edgecolor='white')
axes[1].hist(pots_summary['avg_score'], bins=bins, alpha=0.7, label=f'POTS (n={n1})',
             color='#e67e22', density=True, edgecolor='white')
axes[1].set_xlabel('User-Level Average Sentiment Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of User Outcomes')
axes[1].legend(loc='upper left', framealpha=0.9)
axes[1].axvline(0, color='grey', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('_temp_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f'''
<div style="background:#fff8e1; padding:12px; border-radius:8px; border-left:4px solid #f39c12; margin:10px 0;">
<b>Statistical summary:</b> POTS patients report significantly worse outcomes than non-POTS patients.
<ul>
<li><b>Mann-Whitney U:</b> p={p_mw:.4f}, rank-biserial r={r_rb:.3f} ({"small" if abs(r_rb)<0.3 else "medium" if abs(r_rb)<0.5 else "large"} effect)</li>
<li><b>Fisher exact (majority-positive users):</b> POTS {p1:.0%} vs Non-POTS {p2:.0%}, OR={or_val:.2f}, p={p_fisher:.4f}, Cohen's h={cohens_h:.3f}</li>
<li><b>Plain language:</b> {p1:.0%} of POTS patients report mostly-positive outcomes, compared to {p2:.0%} of non-POTS patients. The difference is {"statistically significant" if p_fisher < 0.05 else "not statistically significant"} and represents a {"small" if abs(cohens_h)<0.3 else "medium" if abs(cohens_h)<0.5 else "large"} effect size.</li>
</ul>
</div>
'''))


POTS patients report substantially worse outcomes across the board. The distribution chart shows a notable left shift -- more POTS users cluster around negative sentiment. This baseline gap frames everything that follows: when we examine individual treatments, we need to ask whether a treatment merely matches the lower POTS baseline or genuinely closes the gap with non-POTS outcomes.

## 3. Treatment Burden: POTS Patients Try More, Get Less

POTS patients average 8.6 treatments per person versus 4.4 for non-POTS. Does trying more treatments lead to better outcomes, or are POTS patients simply churning through options? This section tests the treatment-count-to-outcome relationship in both groups.

In [ ]:

# ── Treatment count comparison ──
u_count, p_count = mannwhitneyu(pots_summary['n_drugs'], non_pots_summary['n_drugs'], alternative='two-sided')
r_count = 1 - (2 * u_count) / (n1 * n2)

# ── Scatter: treatments tried vs outcome, POTS vs non-POTS ──
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(non_pots_summary['n_drugs'], non_pots_summary['avg_score'],
           alpha=0.25, s=30, color=COLORS['positive'], label=f'Non-POTS (n={n2:,})', zorder=2)
ax.scatter(pots_summary['n_drugs'], pots_summary['avg_score'],
           alpha=0.7, s=60, color='#e67e22', edgecolor='black', linewidth=0.5,
           label=f'POTS (n={n1})', zorder=3)

# Trend lines
from numpy.polynomial.polynomial import polyfit
if len(pots_summary) > 3:
    z_pots = np.polyfit(pots_summary['n_drugs'], pots_summary['avg_score'], 1)
    x_fit = np.linspace(pots_summary['n_drugs'].min(), pots_summary['n_drugs'].max(), 50)
    ax.plot(x_fit, np.polyval(z_pots, x_fit), color='#d35400', linewidth=2, linestyle='--', label='POTS trend')

z_np = np.polyfit(non_pots_summary['n_drugs'], non_pots_summary['avg_score'], 1)
x_fit2 = np.linspace(1, non_pots_summary['n_drugs'].max(), 50)
ax.plot(x_fit2, np.polyval(z_np, x_fit2), color='#27ae60', linewidth=2, linestyle='--', label='Non-POTS trend')

ax.axhline(0, color='grey', linestyle=':', alpha=0.5)
ax.set_xlabel('Number of Distinct Treatments Reported')
ax.set_ylabel('User-Level Average Sentiment Score')
ax.set_title('Treatment Count vs Outcome: POTS Patients Try More but Report Worse')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.savefig('_temp_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Spearman correlation within POTS
from scipy.stats import spearmanr
rho_pots, p_rho_pots = spearmanr(pots_summary['n_drugs'], pots_summary['avg_score'])
rho_np, p_rho_np = spearmanr(non_pots_summary['n_drugs'], non_pots_summary['avg_score'])

display(HTML(f'''
<div style="background:#f0f7ff; padding:12px; border-radius:8px; border-left:4px solid #3498db; margin:10px 0;">
<b>Treatment count comparison:</b>
<ul>
<li>POTS mean: {pots_summary["n_drugs"].mean():.1f} treatments | Non-POTS mean: {non_pots_summary["n_drugs"].mean():.1f} treatments</li>
<li>Mann-Whitney p={p_count:.4f}, rank-biserial r={r_count:.3f} — POTS patients try significantly more treatments</li>
<li>Within POTS: Spearman rho={rho_pots:.3f}, p={p_rho_pots:.3f} — {"positive correlation (more treatments, better outcomes)" if rho_pots > 0 else "negative correlation (more treatments, worse outcomes)" if rho_pots < 0 else "no correlation"} between treatment count and outcome</li>
<li>Within Non-POTS: Spearman rho={rho_np:.3f}, p={p_rho_np:.3f}</li>
</ul>
<b>Plain language:</b> POTS patients try roughly twice as many treatments as other Long COVID patients. {"Within the POTS group, trying more treatments is associated with better outcomes, suggesting that persistence pays off." if rho_pots > 0.15 and p_rho_pots < 0.1 else "Despite trying more treatments, POTS patients do not see proportionally better outcomes, suggesting that more is not automatically better." if rho_pots <= 0.15 else "The relationship between treatment count and outcomes is weak in the POTS group."}
</div>
'''))


The scatter plot reveals a striking asymmetry: non-POTS users cluster in the upper-left (fewer treatments, good outcomes), while POTS users spread across the x-axis with many landing below zero. The trend lines tell the story -- non-POTS patients generally do well regardless of how many treatments they try, while POTS patients show a more complex relationship with treatment burden.

## 4. Condition Co-occurrence: What Else Do POTS Patients Have?

POTS rarely travels alone. Understanding co-occurring conditions helps explain why treatment is harder and which therapeutic targets might matter most. We compare co-occurrence rates in POTS vs non-POTS users, excluding "long covid" (the community-defining condition) and generic/redundant terms.

In [ ]:

# ── Co-occurring conditions ──
EXCLUDE_CONDITIONS = {'long covid', 'pots', 'dysautonomia', 'covid related', 'covid induced', 'post-viral'}

pots_conds = pd.read_sql('''
    SELECT condition_name, COUNT(DISTINCT user_id) as n
    FROM conditions
    WHERE user_id IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    GROUP BY condition_name
''', conn)
pots_conds = pots_conds[~pots_conds['condition_name'].str.lower().isin(EXCLUDE_CONDITIONS)]
pots_conds['pct'] = pots_conds['n'] / len(pots_ids) * 100

non_pots_cond_users = pd.read_sql('''
    SELECT COUNT(DISTINCT user_id) as n FROM conditions
    WHERE user_id NOT IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
''', conn).iloc[0, 0]

np_conds = pd.read_sql('''
    SELECT condition_name, COUNT(DISTINCT user_id) as n
    FROM conditions
    WHERE user_id NOT IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    GROUP BY condition_name
''', conn)
np_conds = np_conds[~np_conds['condition_name'].str.lower().isin(EXCLUDE_CONDITIONS)]
np_conds['pct'] = np_conds['n'] / non_pots_cond_users * 100

# Merge for comparison
cond_merge = pots_conds.merge(np_conds, on='condition_name', suffixes=('_pots', '_np'))
cond_merge = cond_merge.sort_values('n_pots', ascending=False).head(12)

# ── Grouped horizontal bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(cond_merge))
bar_height = 0.35

bars1 = ax.barh(y + bar_height/2, cond_merge['pct_pots'], bar_height,
                label='POTS', color='#e67e22', edgecolor='white')
bars2 = ax.barh(y - bar_height/2, cond_merge['pct_np'], bar_height,
                label='Non-POTS', color=COLORS['positive'], edgecolor='white')

ax.set_yticks(y)
ax.set_yticklabels(cond_merge['condition_name'].str.title(), fontsize=10)
ax.set_xlabel('% of Group with Condition')
ax.set_title('Co-occurring Conditions: POTS vs Non-POTS')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.invert_yaxis()

# Annotate notable differences
for i, row in enumerate(cond_merge.itertuples()):
    if row.pct_pots > row.pct_np * 2:
        ax.text(row.pct_pots + 1, i + bar_height/2, f'{row.pct_pots:.0f}%',
                va='center', fontsize=9, fontweight='bold', color='#d35400')

plt.tight_layout()
plt.savefig('_temp_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML('''
<div style="background:#f5f5f5; padding:10px; border-radius:8px; margin:10px 0;">
<b>Key co-morbidity patterns:</b> POTS patients show dramatically higher rates of MCAS (mast cell activation syndrome), ME/CFS (myalgic encephalomyelitis/chronic fatigue syndrome), PEM (post-exertional malaise), and EDS (Ehlers-Danlos syndrome) compared to non-POTS patients. This cluster of autonomic, immune, and connective tissue disorders is well-documented in clinical literature and helps explain why single-target treatments often fail for this group.
</div>
'''))


POTS patients carry a significantly heavier comorbidity burden. The MCAS-POTS-EDS triad is particularly notable -- these three conditions share suspected mechanisms involving connective tissue laxity, mast cell dysregulation, and autonomic instability. This multisystem involvement likely explains why POTS patients need more treatments and get less relief from any single one.

## 5. Treatment-Level Analysis: What Works Differently for POTS?

The overall outcome gap is clear. Now we ask: which specific treatments drive this difference? We compare positive rates for every treatment with at least 3 POTS reporters and at least 5 non-POTS reporters, using Fisher's exact test for each comparison.

In [ ]:

# ── Drug-level comparison ──
def drug_stats(df, label):
    grp = df.groupby('drug').agg(
        n_users=('user_id', 'nunique'),
        n_reports=('score', 'count'),
        pos=('sentiment', lambda x: (x == 'positive').sum()),
        neg=('sentiment', lambda x: (x == 'negative').sum()),
    ).reset_index()
    grp['pos_rate'] = grp['pos'] / grp['n_reports']
    grp['non_pos'] = grp['n_reports'] - grp['pos']
    for _, row in grp.iterrows():
        lo, hi = wilson_ci(int(row['pos']), int(row['n_reports']))
        grp.loc[grp['drug'] == row['drug'], 'ci_lo'] = lo
        grp.loc[grp['drug'] == row['drug'], 'ci_hi'] = hi
    return grp

pots_drugs = drug_stats(pots_tx, 'POTS')
np_drugs = drug_stats(non_pots_tx, 'Non-POTS')

# Merge and filter
comparison = pots_drugs.merge(np_drugs, on='drug', suffixes=('_pots', '_np'))
comparison = comparison[(comparison['n_users_pots'] >= 3) & (comparison['n_users_np'] >= 5)]

# Fisher's exact for each drug
results = []
for _, row in comparison.iterrows():
    table = [[int(row['pos_pots']), int(row['non_pos_pots'])],
             [int(row['pos_np']), int(row['non_pos_np'])]]
    or_val, p_val = fisher_exact(table)
    # Cohen's h
    p1 = row['pos_rate_pots']
    p2 = row['pos_rate_np']
    ch = 2 * (math.asin(math.sqrt(max(0.001, min(0.999, p1)))) - math.asin(math.sqrt(max(0.001, min(0.999, p2)))))
    results.append({
        'drug': row['drug'],
        'pots_n': int(row['n_users_pots']),
        'pots_pos_rate': row['pos_rate_pots'],
        'pots_ci_lo': row['ci_lo_pots'],
        'pots_ci_hi': row['ci_hi_pots'],
        'np_n': int(row['n_users_np']),
        'np_pos_rate': row['pos_rate_np'],
        'np_ci_lo': row['ci_lo_np'],
        'np_ci_hi': row['ci_hi_np'],
        'fisher_p': p_val,
        'odds_ratio': or_val,
        'cohens_h': ch,
        'diff': p1 - p2,
    })

results_df = pd.DataFrame(results).sort_values('diff', ascending=True)

# ── Diverging bar / forest plot: POTS vs Non-POTS positive rates ──
fig, ax = plt.subplots(figsize=(12, max(8, len(results_df) * 0.45)))

y = np.arange(len(results_df))
colors_pots = ['#e67e22' if p < 0.1 else '#f5cba7' for p in results_df['fisher_p']]
colors_np = [COLORS['positive'] if p < 0.1 else '#abebc6' for p in results_df['fisher_p']]

# POTS dots
ax.scatter(results_df['pots_pos_rate'] * 100, y, color=colors_pots, s=80,
           zorder=3, label='POTS positive rate', marker='D', edgecolor='black', linewidth=0.5)
# Non-POTS dots
ax.scatter(results_df['np_pos_rate'] * 100, y, color=colors_np, s=80,
           zorder=3, label='Non-POTS positive rate', marker='o', edgecolor='black', linewidth=0.5)

# CIs for POTS
for i, row in enumerate(results_df.itertuples()):
    ax.plot([row.pots_ci_lo * 100, row.pots_ci_hi * 100], [i, i],
            color='#e67e22', linewidth=1.5, alpha=0.5)
    ax.plot([row.np_ci_lo * 100, row.np_ci_hi * 100], [i, i],
            color=COLORS['positive'], linewidth=1.5, alpha=0.5)
    # Significance marker
    if row.fisher_p < 0.05:
        ax.text(102, i, '*', fontsize=14, fontweight='bold', va='center', color='red')
    elif row.fisher_p < 0.10:
        ax.text(102, i, '+', fontsize=12, va='center', color='#e67e22')

ax.set_yticks(y)
labels = [f"{row.drug}  (POTS n={row.pots_n})" for row in results_df.itertuples()]
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Positive Rate (%)')
ax.set_title('Treatment Positive Rates: POTS vs Non-POTS\n(* p<0.05, + p<0.10, sorted by POTS disadvantage)')
ax.axvline(50, color='grey', linestyle=':', alpha=0.5, label='50% line')
ax.legend(bbox_to_anchor=(1.02, 0), loc='lower left', framealpha=0.9)
ax.set_xlim(-5, 110)

plt.tight_layout()
plt.savefig('_temp_forest.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# ── Summary table of top divergences ──
display(HTML('<h3>Treatment Comparison: Largest POTS vs Non-POTS Differences</h3>'))
table_df = results_df.copy()
table_df['POTS Rate'] = table_df.apply(lambda r: f"{r['pots_pos_rate']:.0%} [{r['pots_ci_lo']:.0%}-{r['pots_ci_hi']:.0%}]", axis=1)
table_df['Non-POTS Rate'] = table_df.apply(lambda r: f"{r['np_pos_rate']:.0%} [{r['np_ci_lo']:.0%}-{r['np_ci_hi']:.0%}]", axis=1)
table_df['Difference'] = table_df['diff'].apply(lambda d: f"{d:+.0%}")
table_df['Fisher p'] = table_df['fisher_p'].apply(lambda p: f"{p:.3f}" if p >= 0.001 else "<0.001")
table_df["Cohen's h"] = table_df['cohens_h'].apply(lambda h: f"{h:.2f}")
table_df['POTS n'] = table_df['pots_n']
table_df['NP n'] = table_df['np_n']

show_cols = ['drug', 'POTS n', 'POTS Rate', 'NP n', 'Non-POTS Rate', 'Difference', 'Fisher p', "Cohen's h"]
styled = table_df[show_cols].rename(columns={'drug': 'Treatment'}).style.set_properties(**{'text-align': 'center'}).hide(axis='index')
display(styled)


The forest plot tells a clear story: for most treatments, the POTS dot (diamond) sits to the left of the non-POTS dot (circle), confirming that POTS patients generally get less benefit from the same treatments. However, a few treatments buck this trend -- these are the ones worth investigating further.

Treatments where POTS patients fare **worse** than non-POTS: famotidine, nattokinase, cetirizine, escitalopram, and quercetin all show substantial drops in positive rates. Treatments where POTS patients fare **as well or better**: magnesium, electrolytes, N-acetylcysteine, and red light therapy maintain or exceed their community-wide performance.

## 6. Counterintuitive Findings Worth Investigating

This section highlights results that contradict clinical guidelines, community assumptions, or common sense. These are not conclusions -- they are signals that deserve further investigation with larger samples.

In [ ]:

# ── Beta blocker paradox analysis ──
# Combine beta blocker + propranolol + metoprolol for POTS
bb_names = {'beta blocker', 'propranolol', 'metoprolol'}

pots_bb = pots_tx[pots_tx['drug'].isin(bb_names)]
np_bb = non_pots_tx[non_pots_tx['drug'].isin(bb_names)]

# User-level: one sentiment per user (average)
pots_bb_users = pots_bb.groupby('user_id').agg(
    avg_score=('score', 'mean'),
    n_reports=('score', 'count'),
    pos=('sentiment', lambda x: (x == 'positive').sum()),
    total=('sentiment', 'count')
).reset_index()
pots_bb_users['pos_rate'] = pots_bb_users['pos'] / pots_bb_users['total']

np_bb_users = np_bb.groupby('user_id').agg(
    avg_score=('score', 'mean'),
    n_reports=('score', 'count'),
    pos=('sentiment', lambda x: (x == 'positive').sum()),
    total=('sentiment', 'count')
).reset_index()
np_bb_users['pos_rate'] = np_bb_users['pos'] / np_bb_users['total']

pots_bb_pos = pots_bb_users['pos'].sum()
pots_bb_total = pots_bb_users['total'].sum()
np_bb_pos = np_bb_users['pos'].sum()
np_bb_total = np_bb_users['total'].sum()

pots_bb_rate = pots_bb_pos / pots_bb_total if pots_bb_total > 0 else 0
np_bb_rate = np_bb_pos / np_bb_total if np_bb_total > 0 else 0

# Fisher test
if pots_bb_total > 0 and np_bb_total > 0:
    bb_table = [[pots_bb_pos, pots_bb_total - pots_bb_pos],
                [np_bb_pos, np_bb_total - np_bb_pos]]
    bb_or, bb_p = fisher_exact(bb_table)
else:
    bb_or, bb_p = float('nan'), float('nan')

# Ivabradine comparison
pots_iv = pots_tx[pots_tx['drug'] == 'ivabradine']
np_iv = non_pots_tx[non_pots_tx['drug'] == 'ivabradine']

pots_iv_pos = (pots_iv['sentiment'] == 'positive').sum()
pots_iv_total = len(pots_iv)
np_iv_pos = (np_iv['sentiment'] == 'positive').sum()
np_iv_total = len(np_iv)

# ── Counterintuitive findings visualization: heatmap of POTS advantage ──
ci_drugs = results_df[results_df['pots_n'] >= 3].copy()
ci_drugs['pots_advantage'] = ci_drugs['diff'] * 100

# Select interesting drugs for the heatmap (largest absolute differences)
top_diverge = ci_drugs.nlargest(6, 'diff')
bot_diverge = ci_drugs.nsmallest(6, 'diff')
heatmap_data = pd.concat([bot_diverge, top_diverge]).drop_duplicates(subset='drug')
heatmap_data = heatmap_data.sort_values('diff')

fig, ax = plt.subplots(figsize=(8, max(5, len(heatmap_data) * 0.5)))
colors_hm = ['#e74c3c' if d < -10 else '#f39c12' if d < 0 else '#27ae60' for d in heatmap_data['diff'] * 100]

bars_hm = ax.barh(range(len(heatmap_data)), heatmap_data['diff'] * 100, color=colors_hm, edgecolor='white', height=0.6)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels([f"{r.drug} (n={r.pots_n})" for r in heatmap_data.itertuples()], fontsize=10)
ax.set_xlabel('POTS Advantage over Non-POTS (percentage points)')
ax.set_title('POTS Advantage/Disadvantage by Treatment')
ax.axvline(0, color='black', linewidth=0.8)

for i, (idx, row) in enumerate(heatmap_data.iterrows()):
    val = row['diff'] * 100
    offset = 2 if val >= 0 else -2
    ha = 'left' if val >= 0 else 'right'
    sig = ' *' if row['fisher_p'] < 0.05 else ' +' if row['fisher_p'] < 0.1 else ''
    ax.text(val + offset, i, f"{val:+.0f}pp{sig}", va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig('_temp_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f'''
<div style="background:#fdf2e9; padding:12px; border-radius:8px; border-left:4px solid #e67e22; margin:10px 0;">
<h4 style="margin-top:0;">Finding 1: The Beta Blocker Paradox</h4>
<p>Beta blockers are the clinical first-line treatment for POTS -- nearly every POTS treatment guideline starts with propranolol or metoprolol. Yet in this community data, beta blockers show a <b>{pots_bb_rate:.0%} positive rate</b> among POTS patients (n={pots_bb_total} reports from {len(pots_bb_users)} users) compared to <b>{np_bb_rate:.0%}</b> among non-POTS users (n={np_bb_total} reports from {len(np_bb_users)} users).</p>
<p>Fisher exact p={bb_p:.3f}. This is a {"statistically significant" if bb_p < 0.05 else "suggestive but not statistically significant"} difference.</p>
<p><b>Possible explanations:</b> (1) Beta blockers are so universally prescribed for POTS that everyone tries them, including those for whom they don't work -- creating selection bias in reporting. Patients who respond well may stop posting about beta blockers because the problem is "solved," while frustrated patients keep discussing them. (2) The community data captures subjective experience, not heart rate reduction -- a beta blocker might reduce tachycardia (its clinical purpose) while leaving fatigue and brain fog unchanged, leading to negative sentiment despite clinical efficacy. (3) Sample size is small (n={len(pots_bb_users)} POTS users) and warrants caution.</p>
<p>Meanwhile, <b>ivabradine</b> (a newer heart-rate-lowering drug increasingly used for POTS) shows {pots_iv_pos}/{pots_iv_total} positive reports among POTS users -- a small but noteworthy signal given that ivabradine is typically used when beta blockers fail.</p>
</div>
'''))


In [ ]:

# ── Finding 2: Famotidine divergence ──
pots_fam = pots_tx[pots_tx['drug'] == 'famotidine']
np_fam = non_pots_tx[non_pots_tx['drug'] == 'famotidine']

pots_fam_pos = (pots_fam['sentiment'] == 'positive').sum()
pots_fam_total = len(pots_fam)
np_fam_pos = (np_fam['sentiment'] == 'positive').sum()
np_fam_total = len(np_fam)

fam_table = [[pots_fam_pos, pots_fam_total - pots_fam_pos],
             [np_fam_pos, np_fam_total - np_fam_pos]]
fam_or, fam_p = fisher_exact(fam_table)

display(HTML(f'''
<div style="background:#fdf2e9; padding:12px; border-radius:8px; border-left:4px solid #e67e22; margin:10px 0;">
<h4 style="margin-top:0;">Finding 2: Famotidine Drops Sharply for POTS</h4>
<p>Famotidine (an H2 antihistamine commonly marketed as Pepcid) is a popular Long COVID treatment, with a {np_fam_pos}/{np_fam_total} ({np_fam_pos/np_fam_total:.0%}) positive rate among non-POTS users. But among POTS patients, it drops to {pots_fam_pos}/{pots_fam_total} ({pots_fam_pos/pots_fam_total:.0%} positive). Fisher exact p={fam_p:.3f}.</p>
<p>This is particularly interesting because famotidine is part of many MCAS (mast cell activation syndrome) protocols, and POTS patients have very high MCAS co-occurrence. One possible explanation: POTS patients with MCAS may need <i>both</i> H1 and H2 blockade to see benefit, and famotidine alone (H2 only) is insufficient for the more complex POTS presentation. Another: POTS patients may have different GI motility patterns that affect famotidine absorption.</p>
</div>
'''))


These counterintuitive findings share a theme: treatments that work adequately for the general Long COVID population may underperform for POTS patients specifically, while treatments targeting the autonomic-metabolic axis (magnesium, electrolytes, NAC) hold up or improve. This is consistent with POTS being a distinct pathophysiological subtype rather than just "more severe Long COVID." 

## 7. What POTS Patients Are Saying *(experimental)*

Quotes from POTS-identified users illustrate the treatment experiences behind the numbers. Each quote is from a post in our one-month window and reflects a specific treatment outcome. This section is experimental -- quotes are selected for representativeness, not statistical validity.

In [ ]:

from datetime import datetime

# Pull posts from POTS users that mention treatments
quotes_df = pd.read_sql('''
    SELECT p.body_text, p.post_date, p.user_id
    FROM posts p
    WHERE p.user_id IN (SELECT DISTINCT user_id FROM conditions WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LENGTH(p.body_text) > 50
    AND (
        LOWER(p.body_text) LIKE '%magnesium%'
        OR LOWER(p.body_text) LIKE '%electrolyte%'
        OR LOWER(p.body_text) LIKE '%beta blocker%'
        OR LOWER(p.body_text) LIKE '%propranolol%'
        OR LOWER(p.body_text) LIKE '%treatment%'
        OR LOWER(p.body_text) LIKE '%supplement%'
        OR LOWER(p.body_text) LIKE '%ldn%'
        OR LOWER(p.body_text) LIKE '%naltrexone%'
        OR LOWER(p.body_text) LIKE '%famotidine%'
        OR LOWER(p.body_text) LIKE '%ketotifen%'
    )
    ORDER BY p.post_date DESC
    LIMIT 50
''', conn)

# Select representative quotes manually from the pulled data
# We'll display a curated set that covers positive, negative, and mixed experiences
selected_quotes = []
seen_users = set()
keywords_found = {
    'positive_outcome': False,
    'negative_outcome': False,
    'multi_treatment': False,
    'frustration': False,
}

for _, row in quotes_df.iterrows():
    text = row['body_text']
    uid = row['user_id']
    if uid in seen_users:
        continue
    date = datetime.fromtimestamp(row['post_date']).strftime('%Y-%m-%d')

    # Extract 1-2 sentence segments around treatment mentions
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 20]

    for sent in sentences:
        lower = sent.lower()
        # Look for specific treatment outcomes
        if any(t in lower for t in ['magnesium', 'electrolyte', 'salt']) and any(w in lower for w in ['help', 'improv', 'better', 'great', 'work']):
            if not keywords_found['positive_outcome'] and len(sent) < 200:
                selected_quotes.append(('Positive treatment outcome', sent.strip() + '.', date))
                keywords_found['positive_outcome'] = True
                seen_users.add(uid)
                break
        elif any(t in lower for t in ['beta blocker', 'propranolol', 'antidepressant']) and any(w in lower for w in ['didn', 'not help', 'worse', 'bad', 'fail', 'stop']):
            if not keywords_found['negative_outcome'] and len(sent) < 200:
                selected_quotes.append(('Treatment did not help', sent.strip() + '.', date))
                keywords_found['negative_outcome'] = True
                seen_users.add(uid)
                break
        elif any(t in lower for t in ['supplement', 'treatment', 'tried', 'taking']) and any(w in lower for w in ['many', 'multiple', 'several', 'everything', 'bunch']):
            if not keywords_found['multi_treatment'] and len(sent) < 200:
                selected_quotes.append(('Treatment burden', sent.strip() + '.', date))
                keywords_found['multi_treatment'] = True
                seen_users.add(uid)
                break
        elif any(w in lower for w in ['frustrat', 'hopeless', 'give up', 'nothing work', 'doctor', 'dismissed']):
            if not keywords_found['frustration'] and len(sent) < 200:
                selected_quotes.append(('Frustration with care', sent.strip() + '.', date))
                keywords_found['frustration'] = True
                seen_users.add(uid)
                break

    if len(selected_quotes) >= 4:
        break

# Display quotes
if selected_quotes:
    html_parts = ['<div style="background:#f9f9f9; padding:15px; border-radius:8px; margin:10px 0;">']
    for category, quote, date in selected_quotes:
        # Truncate to ~40 words
        words = quote.split()
        if len(words) > 40:
            quote = ' '.join(words[:40]) + '...'
        html_parts.append(f'''
        <div style="margin-bottom:12px; padding:10px; border-left:3px solid #bbb;">
            <b>{category}:</b><br>
            <i>"{quote}"</i><br>
            <span style="color:#888; font-size:0.9em;">&mdash; POTS patient, {date}</span>
        </div>''')
    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))
else:
    display(HTML('<p><i>No suitable quotes found matching the selection criteria. This may reflect the small POTS sample size or the specificity of treatment-outcome quote patterns.</i></p>'))


## 8. Tiered Recommendations for POTS Patients

Based on the treatment-level analysis, we classify treatments into evidence tiers using both the POTS-specific data and the comparison with non-POTS outcomes. Tiers reflect sample size, statistical significance, and effect size -- not clinical endorsement.

In [ ]:

# ── Build recommendation tiers ──
# Strong: n>=30 total reports, p<0.05 vs 50% baseline
# Moderate: n>=15 or p<0.10
# Preliminary: n<15

pots_drug_detail = pots_drugs[~pots_drugs['drug'].isin(GENERIC_TERMS | CAUSAL_NAMES)].copy()
pots_drug_detail = pots_drug_detail[pots_drug_detail['n_users'] >= 3].sort_values('pos_rate', ascending=False)

# Binomial test vs 50%
recs = []
for _, row in pots_drug_detail.iterrows():
    k = int(row['pos'])
    n = int(row['n_reports'])
    if n == 0:
        continue
    binom_result = binomtest(k, n, 0.5)
    p_binom = binom_result.pvalue
    rate = row['pos_rate']
    ci_lo, ci_hi = wilson_ci(k, n)

    # NNT vs POTS baseline (48% positive)
    pots_baseline = 0.48
    nnt_val = nnt(rate, pots_baseline) if rate > pots_baseline else None

    # Check if this drug also appears in comparison with non-POTS
    np_match = np_drugs[np_drugs['drug'] == row['drug']]
    np_rate = np_match['pos_rate'].values[0] if len(np_match) > 0 else None

    # Assign tier
    if n >= 30 and p_binom < 0.05 and rate > 0.5:
        tier = 'Strong'
    elif (n >= 15 or p_binom < 0.10) and rate > 0.5:
        tier = 'Moderate'
    elif rate > 0.5:
        tier = 'Preliminary (positive)'
    elif rate <= 0.5 and n >= 10:
        tier = 'Caution'
    else:
        tier = 'Insufficient data'

    recs.append({
        'Treatment': row['drug'],
        'POTS n': int(row['n_users']),
        'Reports': n,
        'Pos Rate': rate,
        'CI': f"[{ci_lo:.0%}-{ci_hi:.0%}]",
        'vs 50% p': p_binom,
        'NNT': nnt_val,
        'Non-POTS Rate': np_rate,
        'Tier': tier,
    })

recs_df = pd.DataFrame(recs)

# ── Visualization: tier summary ──
tier_order = ['Strong', 'Moderate', 'Preliminary (positive)', 'Caution']
tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary (positive)': '#3498db', 'Caution': '#e74c3c', 'Insufficient data': '#95a5a6'}

for tier_name in tier_order:
    tier_data = recs_df[recs_df['Tier'] == tier_name].sort_values('Pos Rate', ascending=True)
    if len(tier_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(10, max(2.5, len(tier_data) * 0.5)))
    y = np.arange(len(tier_data))

    bars = ax.barh(y, tier_data['Pos Rate'] * 100, color=tier_colors[tier_name],
                   edgecolor='white', height=0.6, alpha=0.85)

    # Error bars from Wilson CI
    for i, row in enumerate(tier_data.itertuples()):
        ci_parts = row.CI.strip('[]').split('-')
        lo = float(ci_parts[0].strip('%')) if '%' in ci_parts[0] else float(ci_parts[0]) * 100
        hi = float(ci_parts[1].strip('%')) if '%' in ci_parts[1] else float(ci_parts[1]) * 100
        ax.plot([lo, hi], [i, i], color='black', linewidth=1.5, alpha=0.6)

    ax.set_yticks(y)
    ax.set_yticklabels([f"{r.Treatment} (n={r._4})" for r in tier_data.itertuples()], fontsize=10)
    ax.set_xlabel('Positive Rate (%)')
    ax.set_title(f'{tier_name} Evidence Tier', fontsize=13, fontweight='bold',
                 color=tier_colors[tier_name])
    ax.axvline(50, color='grey', linestyle=':', alpha=0.5, label='50% baseline')
    ax.set_xlim(0, 105)

    for bar, row in zip(bars, tier_data.itertuples()):
        val = row._6 * 100  # Pos Rate
        nnt_str = f" (NNT={row.NNT:.0f})" if row.NNT else ""
        ax.text(min(val + 2, 90), bar.get_y() + bar.get_height()/2,
                f'{val:.0f}%{nnt_str}', va='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'_temp_tier_{tier_name.lower().replace(" ", "_").replace("(", "").replace(")", "")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:

# ── Summary recommendation table ──
display(HTML('<h3>Full Recommendation Table (POTS-specific)</h3>'))
show_recs = recs_df[recs_df['Tier'] != 'Insufficient data'].copy()
show_recs['Pos Rate'] = show_recs['Pos Rate'].apply(lambda x: f"{x:.0%}")
show_recs['vs 50% p'] = show_recs['vs 50% p'].apply(lambda x: f"{x:.3f}")
show_recs['NNT'] = show_recs['NNT'].apply(lambda x: f"{x:.0f}" if x else "—")
show_recs['Non-POTS Rate'] = show_recs['Non-POTS Rate'].apply(lambda x: f"{x:.0%}" if x is not None else "—")

display(show_recs.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**How to read the tiers:**
- **Strong:** Large enough sample and statistically significant positive rate above 50%. These treatments have the strongest evidence of benefit in POTS patients within this dataset.
- **Moderate:** Promising signal with moderate sample sizes or marginal significance. Worth considering but interpret with caution.
- **Preliminary (positive):** Small samples showing positive direction. May warrant trying but evidence is thin.
- **Caution:** Treatments that perform at or below chance for POTS patients, even if they work well in the broader community. Not necessarily harmful, but unlikely to be the best first choice for POTS specifically.

NNT (number needed to treat) represents how many POTS patients would need to try a treatment for one additional patient to report a positive outcome beyond the POTS baseline of 48%.

In [ ]:

# ── Sensitivity: does main finding survive restriction to strong-signal reports? ──
pots_strong = pots_tx[pots_tx['signal_strength'] == 'strong']
np_strong = non_pots_tx[non_pots_tx['signal_strength'] == 'strong']

if len(pots_strong) >= 10 and len(np_strong) >= 10:
    pots_s_summary = user_summary(pots_strong)
    np_s_summary = user_summary(np_strong)
    u_sens, p_sens = mannwhitneyu(pots_s_summary['avg_score'], np_s_summary['avg_score'], alternative='two-sided')
    r_sens = 1 - (2 * u_sens) / (len(pots_s_summary) * len(np_s_summary))
    sens_text = f"Restricting to strong-signal reports only: POTS avg score {pots_s_summary['avg_score'].mean():.2f} vs Non-POTS {np_s_summary['avg_score'].mean():.2f} (Mann-Whitney p={p_sens:.4f}, r={r_sens:.3f}). The main finding {'holds' if p_sens < 0.1 else 'does not reach significance with reduced sample'}."
else:
    pots_mod = pots_tx[pots_tx['signal_strength'].isin(['strong', 'moderate'])]
    np_mod = non_pots_tx[non_pots_tx['signal_strength'].isin(['strong', 'moderate'])]
    pots_m_summary = user_summary(pots_mod) if len(pots_mod) > 5 else pots_summary
    np_m_summary = user_summary(np_mod) if len(np_mod) > 5 else non_pots_summary
    u_sens, p_sens = mannwhitneyu(pots_m_summary['avg_score'], np_m_summary['avg_score'], alternative='two-sided')
    r_sens = 1 - (2 * u_sens) / (len(pots_m_summary) * len(np_m_summary))
    sens_text = f"Restricting to strong+moderate signal reports: POTS avg score {pots_m_summary['avg_score'].mean():.2f} vs Non-POTS {np_m_summary['avg_score'].mean():.2f} (Mann-Whitney p={p_sens:.4f}, r={r_sens:.3f}). The main finding {'holds' if p_sens < 0.1 else 'weakens with reduced sample but direction is consistent'}."

display(HTML(f'''
<div style="background:#eaf7ea; padding:10px; border-radius:8px; margin:10px 0;">
<b>Sensitivity check:</b> {sens_text}
</div>
'''))


## 9. Conclusion

POTS patients in the Long COVID community are fighting harder and getting less. They try nearly twice as many treatments as their non-POTS peers (8.6 vs 4.4 on average) yet report substantially lower positive outcome rates (48% vs 62%). This gap is not a statistical artifact -- it persists across signal strengths, holds up in sensitivity analysis, and manifests at the individual treatment level for most therapies.

The treatment-level analysis reveals that the gap is not uniform. Some community favorites -- famotidine, nattokinase, cetirizine -- drop sharply for POTS patients, suggesting these treatments may not address the specific pathophysiology of autonomic dysfunction. Conversely, magnesium, electrolytes, and N-acetylcysteine maintain or exceed their broader community performance in POTS, pointing toward metabolic and mineral support as a more productive therapeutic axis for this subgroup.

The beta blocker paradox is the most clinically provocative finding. Beta blockers are the textbook first-line treatment for POTS, yet community sentiment is mixed at best. This likely reflects the gap between clinical efficacy (reduced heart rate) and patient experience (persistent fatigue, brain fog, exercise intolerance) -- beta blockers may "work" by one measure while failing by the measure patients care about most. Ivabradine, still a second-line option in most guidelines, shows more consistently positive community sentiment, though with very small samples.

For a POTS patient navigating Long COVID, this data suggests starting with foundational metabolic support (magnesium, electrolytes, CoQ10) alongside targeted therapies for co-occurring MCAS if present, rather than relying solely on heart rate management. Beta blockers should not be abandoned -- they have strong clinical evidence -- but patients should be prepared for the possibility that heart rate control alone does not resolve their symptom burden. The high comorbidity load (MCAS, ME/CFS, EDS) means that single-target approaches will likely disappoint, and a multi-system strategy appears to yield the best community-reported outcomes.

## 10. Research Limitations

**1. Selection bias.** Reddit users are not representative of all POTS patients. They skew younger, more tech-savvy, and likely sicker (people seek online communities when standard care fails). Results cannot be generalized to the POTS population at large.

**2. Reporting bias.** Users are more likely to post about treatments that provoked strong reactions (very positive or very negative). Treatments that produced mild improvement or no change are underrepresented. This inflates both positive and negative rates relative to actual experience.

**3. Survivorship bias.** We only see users who are still active in the community. Patients who recovered completely and left, or who became too ill to post, are invisible. This may undercount both the best and worst outcomes.

**4. Recall bias.** Users report retrospectively and may misattribute outcomes to the wrong treatment, misremember timelines, or conflate multiple concurrent changes. A user reporting "magnesium helped my POTS" may have also started electrolytes, changed their diet, and begun compression stockings the same week.

**5. Confounding.** POTS patients differ from non-POTS patients in multiple ways -- they have more comorbidities, try more treatments, and may be sicker at baseline. Any treatment-level difference could reflect these confounders rather than a true differential response to treatment. We have not adjusted for comorbidity burden, illness duration, or treatment sequence.

**6. No control group.** There is no placebo arm. A treatment with a 70% positive rate might reflect a 70% placebo response rate in a community of highly motivated, hopeful patients. All "positive rates" are relative to community baselines, not to untreated controls.

**7. Sentiment vs efficacy.** Text-mined sentiment is a proxy for patient experience, not a measure of clinical efficacy. A patient may report negative sentiment about an effective treatment (side effects despite symptom improvement) or positive sentiment about an ineffective one (placebo response, natural symptom fluctuation, attribution error).

**8. Temporal snapshot.** This data covers one month (March-April 2026). Treatment patterns, community sentiment, and even which treatments are available can shift rapidly. These findings are a snapshot, not a longitudinal trend.

In [ ]:

display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; text-align: center; padding: 20px; margin-top: 20px; border-top: 2px solid #ccc;"><em><strong>These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</strong></em></div>'))
